# 🎬 AI-Based Video Fact Checker
This notebook implements the pipeline described in the project:

1. Upload a video
2. Extract audio → transcribe with **Whisper**
3. Extract key claims/highlights with **NLP** (summarization + keyword extraction)
4. Search **NewsAPI** for corroborating articles (once, using overall video content — not per sentence)
5. Compute an **authenticity/risk score** (source credibility + semantic similarity to news)
6. Run a **basic deepfake/video-tamper heuristic** (frame-level face & compression consistency)
7. Store results in **SQLite**
8. Generate a **PDF verification report**
9. Wrap everything in a simple **Gradio** web UI

> ⚠️ **Honesty note on deepfake detection:** true deepfake detection needs a trained forensic model (e.g. a CNN trained on FaceForensics++). Training/hosting that is out of scope for a Colab demo, so Step 6 implements a *lightweight heuristic* (face-region blur/noise consistency across frames) that gives a rough "visual risk" signal — label it as such in your report/demo, not as a certified deepfake detector. If you have access to a pretrained deepfake-detection model/weights, swap it into the `analyze_video_frames()` function.

Run the cells top to bottom. You'll need a free **NewsAPI** key from https://newsapi.org (Cell 3).


## 1. Install dependencies

In [ ]:
!pip install -q openai-whisper moviepy opencv-python-headless fpdf2 \
    sentence-transformers yake requests gradio nltk transformers accelerate

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ Dependencies installed")


## 2. Imports & project setup

In [ ]:
import os, sqlite3, json, datetime, textwrap, cv2, requests, numpy as np
import whisper
from moviepy.editor import VideoFileClip
from fpdf import FPDF
from sentence_transformers import SentenceTransformer, util
import yake
from transformers import pipeline

PROJECT_DIR = "/content/video_fact_checker"
os.makedirs(PROJECT_DIR, exist_ok=True)
DB_PATH = os.path.join(PROJECT_DIR, "fact_checker.db")

print("✅ Imports ready. Project folder:", PROJECT_DIR)


## 3. Configuration
Paste your free NewsAPI key below (get one at https://newsapi.org/register).


In [ ]:
NEWS_API_KEY = "YOUR_NEWSAPI_KEY_HERE"   # <-- replace this
WHISPER_MODEL_SIZE = "base"   # tiny / base / small / medium / large (bigger = slower but more accurate)

assert NEWS_API_KEY and NEWS_API_KEY != "YOUR_NEWSAPI_KEY_HERE", \
    "Please set your NewsAPI key above before continuing."


## 4. Upload video

In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = os.path.join(PROJECT_DIR, list(uploaded.keys())[0])
with open(VIDEO_PATH, "wb") as f:
    f.write(list(uploaded.values())[0])

print("✅ Uploaded:", VIDEO_PATH)


## 5. Extract audio from video

In [ ]:
def extract_audio(video_path, out_dir=PROJECT_DIR):
    clip = VideoFileClip(video_path)
    audio_path = os.path.join(out_dir, "audio.wav")
    clip.audio.write_audiofile(audio_path, fps=16000, verbose=False, logger=None)
    duration = clip.duration
    clip.close()
    return audio_path, duration

AUDIO_PATH, VIDEO_DURATION = extract_audio(VIDEO_PATH)
print(f"✅ Audio extracted: {AUDIO_PATH} ({VIDEO_DURATION:.1f}s)")


## 6. Transcribe speech with Whisper

In [ ]:
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path)
    return result["text"].strip(), result.get("segments", [])

TRANSCRIPT, SEGMENTS = transcribe_audio(AUDIO_PATH)
print("✅ Transcript length:", len(TRANSCRIPT.split()), "words")
print(textwrap.shorten(TRANSCRIPT, width=500, placeholder=" ..."))


## 7. NLP: extract key information / highlights
Two things are produced:
- a short **summary** of the whole transcript (used to query NewsAPI once, not sentence-by-sentence)
- a list of **keywords/key phrases** (used to refine the news search and shown to the user as "key claims")


In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
kw_extractor = yake.KeywordExtractor(lan="en", n=3, top=8)

def chunk_text(text, max_words=500):
    words = text.split()
    for i in range(0, len(words), max_words):
        yield " ".join(words[i:i + max_words])

def summarize_transcript(text):
    if len(text.split()) < 40:
        return text  # too short to summarize meaningfully
    chunks = list(chunk_text(text, 500))
    partial_summaries = []
    for chunk in chunks:
        max_len = min(120, max(30, len(chunk.split()) // 2))
        summary = summarizer(chunk, max_length=max_len, min_length=15, do_sample=False)[0]["summary_text"]
        partial_summaries.append(summary)
    combined = " ".join(partial_summaries)
    if len(chunks) > 1 and len(combined.split()) > 60:
        combined = summarizer(combined, max_length=100, min_length=25, do_sample=False)[0]["summary_text"]
    return combined.strip()

def extract_keywords(text):
    keywords = kw_extractor.extract_keywords(text)
    return [kw for kw, score in sorted(keywords, key=lambda x: x[1])]

SUMMARY = summarize_transcript(TRANSCRIPT)
KEYWORDS = extract_keywords(TRANSCRIPT)

print("📌 SUMMARY:\n", SUMMARY)
print("\n📌 KEY TERMS:\n", KEYWORDS)


## 8. Search NewsAPI for corroborating sources
Uses the overall video summary + top keywords as **one combined query**, instead of
searching separately per sentence — matching the design goal of the project.


In [ ]:
def search_news(summary, keywords, api_key, page_size=8):
    query_terms = keywords[:5] if keywords else summary.split()[:8]
    query = " OR ".join(f'"{t}"' if " " in t else t for t in query_terms)

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "language": "en",
        "sortBy": "relevancy",
        "pageSize": page_size,
        "apiKey": api_key,
    }
    resp = requests.get(url, params=params, timeout=15)
    if resp.status_code != 200:
        print("⚠️ NewsAPI error:", resp.status_code, resp.text[:300])
        return []

    articles = resp.json().get("articles", [])
    return [
        {
            "title": a.get("title"),
            "source": (a.get("source") or {}).get("name"),
            "url": a.get("url"),
            "publishedAt": a.get("publishedAt"),
            "description": a.get("description") or "",
        }
        for a in articles
    ]

NEWS_RESULTS = search_news(SUMMARY, KEYWORDS, NEWS_API_KEY)
print(f"✅ Found {len(NEWS_RESULTS)} related articles")
for a in NEWS_RESULTS[:5]:
    print(f"- {a['title']}  ({a['source']})")


## 9. Authenticity / risk scoring
Combines two signals:
- **Semantic similarity** between the video's summary and each article (via sentence-transformers)
- **Source credibility** — a small illustrative whitelist; replace/extend with your own vetted list for production use

Produces a 0–100 "corroboration score" and a plain-language verdict band. This is a heuristic
support tool, **not** a definitive fact-checking verdict.


In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# Illustrative credibility tiers - customize as needed for your use case
CREDIBLE_SOURCES = {
    "Reuters", "Associated Press", "BBC News", "The Wall Street Journal",
    "The New York Times", "NPR", "The Guardian", "Bloomberg", "Al Jazeera English",
}

def compute_authenticity(summary, articles):
    if not articles:
        return {
            "score": 0,
            "verdict": "Unverifiable - no matching news coverage found",
            "matched_articles": [],
        }

    summary_emb = sbert_model.encode(summary, convert_to_tensor=True)
    scored = []
    for a in articles:
        text = f"{a['title']}. {a['description']}"
        emb = sbert_model.encode(text, convert_to_tensor=True)
        sim = float(util.cos_sim(summary_emb, emb)[0][0])
        credibility_bonus = 10 if a["source"] in CREDIBLE_SOURCES else 0
        scored.append({**a, "similarity": round(sim, 3), "credibility_bonus": credibility_bonus})

    scored.sort(key=lambda x: x["similarity"], reverse=True)
    top = scored[:5]
    avg_sim = sum(x["similarity"] for x in top) / len(top)
    credibility_boost = sum(x["credibility_bonus"] for x in top) / len(top)
    raw_score = (avg_sim * 80) + credibility_boost
    score = max(0, min(100, round(raw_score)))

    if score >= 70:
        verdict = "Likely accurate - strong corroboration from news sources"
    elif score >= 40:
        verdict = "Partially supported - some related coverage, verify details"
    else:
        verdict = "Low corroboration - little matching coverage, treat with caution"

    return {"score": score, "verdict": verdict, "matched_articles": top}

AUTHENTICITY = compute_authenticity(SUMMARY, NEWS_RESULTS)
print(f"🧾 Authenticity score: {AUTHENTICITY['score']}/100")
print("🧾 Verdict:", AUTHENTICITY["verdict"])


## 10. Basic video-tamper / deepfake heuristic
Samples frames, runs face detection, and checks for inconsistencies in
blur level and noise pattern across detected faces — abrupt jumps can indicate splicing
or synthetic frames. This is a **lightweight indicator**, not a forensic-grade classifier.


In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def analyze_video_frames(video_path, sample_every_n_sec=1.0, max_samples=40):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = max(1, int(fps * sample_every_n_sec))

    blur_scores, face_found_flags = [], []
    frame_idx, samples_taken = 0, 0

    while cap.isOpened() and samples_taken < max_samples:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 5)
            face_found_flags.append(len(faces) > 0)
            if len(faces) > 0:
                x, y, w, h = faces[0]
                face_roi = gray[y:y + h, x:x + w]
                if face_roi.size > 0:
                    blur = cv2.Laplacian(face_roi, cv2.CV_64F).var()
                    blur_scores.append(blur)
            samples_taken += 1
        frame_idx += 1
    cap.release()

    if len(blur_scores) < 3:
        return {"risk_score": None, "note": "Not enough face frames detected to assess."}

    blur_arr = np.array(blur_scores)
    variability = float(np.std(blur_arr) / (np.mean(blur_arr) + 1e-6))  # coefficient of variation
    face_consistency = sum(face_found_flags) / len(face_found_flags)

    # Heuristic: high sharpness variability + inconsistent face presence -> higher "visual risk"
    risk = min(100, round(variability * 120 + (1 - face_consistency) * 40))

    if risk >= 60:
        note = "High visual inconsistency detected — recommend manual review for tampering."
    elif risk >= 30:
        note = "Moderate visual inconsistency — inconclusive, review recommended."
    else:
        note = "Low visual inconsistency — no strong signs of tampering detected (not a guarantee of authenticity)."

    return {"risk_score": risk, "note": note, "faces_detected_ratio": round(face_consistency, 2)}

DEEPFAKE_ANALYSIS = analyze_video_frames(VIDEO_PATH)
print("🎭 Visual risk analysis:", DEEPFAKE_ANALYSIS)


## 11. Store results in SQLite

In [ ]:
def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_name TEXT,
            created_at TEXT,
            transcript TEXT,
            summary TEXT,
            keywords TEXT,
            authenticity_score INTEGER,
            verdict TEXT,
            visual_risk_score INTEGER,
            visual_note TEXT,
            matched_articles_json TEXT
        )
    ''')
    conn.commit()
    return conn

def save_report(video_path, transcript, summary, keywords, authenticity, deepfake):
    conn = init_db()
    conn.execute('''
        INSERT INTO reports (
            video_name, created_at, transcript, summary, keywords,
            authenticity_score, verdict, visual_risk_score, visual_note, matched_articles_json
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        os.path.basename(video_path),
        datetime.datetime.now().isoformat(),
        transcript,
        summary,
        json.dumps(keywords),
        authenticity["score"],
        authenticity["verdict"],
        deepfake.get("risk_score"),
        deepfake.get("note"),
        json.dumps(authenticity["matched_articles"]),
    ))
    conn.commit()
    report_id = conn.execute("SELECT last_insert_rowid()").fetchone()[0]
    conn.close()
    return report_id

REPORT_ID = save_report(VIDEO_PATH, TRANSCRIPT, SUMMARY, KEYWORDS, AUTHENTICITY, DEEPFAKE_ANALYSIS)
print(f"✅ Saved to SQLite as report #{REPORT_ID} ({DB_PATH})")


## 12. Generate PDF verification report

In [ ]:
def clean_text(t):
    return t.encode("latin-1", "replace").decode("latin-1")

def generate_pdf_report(report_id, video_name, summary, keywords, authenticity, deepfake, out_dir=PROJECT_DIR):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, "Video Fact-Check Report", ln=True)
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 8, f"Report ID: {report_id}   |   Generated: {datetime.datetime.now():%Y-%m-%d %H:%M}", ln=True)
    pdf.cell(0, 8, f"Source video: {clean_text(video_name)}", ln=True)
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 8, "Summary of Content", ln=True)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, clean_text(summary))
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 8, "Key Terms", ln=True)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, clean_text(", ".join(keywords)))
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 8, "Authenticity Assessment", ln=True)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, clean_text(f"Score: {authenticity['score']}/100 - {authenticity['verdict']}"))
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "Matched News Sources", ln=True)
    pdf.set_font("Helvetica", "", 10)
    if authenticity["matched_articles"]:
        for a in authenticity["matched_articles"]:
            pdf.multi_cell(0, 5, clean_text(f"- {a['title']} ({a['source']}) - similarity {a['similarity']}"))
            pdf.set_text_color(0, 0, 255)
            pdf.multi_cell(0, 5, clean_text(a["url"]))
            pdf.set_text_color(0, 0, 0)
            pdf.ln(1)
    else:
        pdf.multi_cell(0, 6, "No corroborating articles found.")
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 8, "Visual / Deepfake Risk Indicator", ln=True)
    pdf.set_font("Helvetica", "", 11)
    risk_txt = f"Risk score: {deepfake.get('risk_score')}/100 - {deepfake.get('note')}"
    pdf.multi_cell(0, 6, clean_text(risk_txt))
    pdf.set_font("Helvetica", "I", 9)
    pdf.multi_cell(0, 5, "Note: heuristic indicator only, not a certified forensic result.")

    out_path = os.path.join(out_dir, f"fact_check_report_{report_id}.pdf")
    pdf.output(out_path)
    return out_path

PDF_PATH = generate_pdf_report(REPORT_ID, os.path.basename(VIDEO_PATH), SUMMARY, KEYWORDS, AUTHENTICITY, DEEPFAKE_ANALYSIS)
print("✅ PDF report generated:", PDF_PATH)

from google.colab import files as colab_files
colab_files.download(PDF_PATH)


## 13. End-to-end pipeline function
Wraps steps 5–12 into a single function, so the Gradio UI (next cell) or future videos
can be processed with one call.


In [ ]:
def run_pipeline(video_path, news_api_key=NEWS_API_KEY):
    audio_path, _ = extract_audio(video_path)
    transcript, _ = transcribe_audio(audio_path)
    summary = summarize_transcript(transcript)
    keywords = extract_keywords(transcript)
    news_results = search_news(summary, keywords, news_api_key)
    authenticity = compute_authenticity(summary, news_results)
    deepfake = analyze_video_frames(video_path)
    report_id = save_report(video_path, transcript, summary, keywords, authenticity, deepfake)
    pdf_path = generate_pdf_report(report_id, os.path.basename(video_path), summary, keywords, authenticity, deepfake)
    return {
        "report_id": report_id,
        "transcript": transcript,
        "summary": summary,
        "keywords": keywords,
        "authenticity": authenticity,
        "deepfake": deepfake,
        "pdf_path": pdf_path,
    }

print("✅ Pipeline function ready: run_pipeline(video_path)")


## 14. Web UI with Gradio

In [ ]:
import gradio as gr

def gradio_handler(video_file):
    result = run_pipeline(video_file)
    articles_md = "\n".join(
        f"- [{a['title']}]({a['url']}) — *{a['source']}* (similarity: {a['similarity']})"
        for a in result["authenticity"]["matched_articles"]
    ) or "No matching articles found."

    summary_md = f"""
### 📝 Summary
{result['summary']}

### 🔑 Key Terms
{', '.join(result['keywords'])}

### ✅ Authenticity Score: {result['authenticity']['score']}/100
**Verdict:** {result['authenticity']['verdict']}

### 🎭 Visual Risk Indicator: {result['deepfake'].get('risk_score')}/100
{result['deepfake'].get('note')}

### 📰 Matched Sources
{articles_md}
"""
    return summary_md, result["pdf_path"]

demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Video(label="Upload a video to fact-check"),
    outputs=[gr.Markdown(label="Report"), gr.File(label="Download PDF Report")],
    title="AI-Based Video Fact Checker",
    description="Upload a video. The system transcribes it, extracts key claims, checks them against news coverage, and flags visual inconsistencies.",
)

demo.launch(share=True, debug=False)


## Notes & next steps
- **NewsAPI free tier** limits requests/day and only returns articles from the last month — fine for a demo, upgrade for production.
- **Deepfake detection**: swap `analyze_video_frames()` for a real pretrained model (e.g. from FaceForensics++, or a HuggingFace deepfake-detection checkpoint) for meaningfully stronger signal.
- **Source credibility list**: replace `CREDIBLE_SOURCES` with a maintained, well-documented methodology (e.g. NewsGuard, Media Bias/Fact Check ratings) rather than a hardcoded whitelist for anything beyond a demo.
- **Scaling**: for longer videos, consider chunked Whisper transcription with timestamps so the PDF report can cite *where* in the video each claim appears.
- **Persistence across sessions**: Colab's filesystem resets when the runtime disconnects — copy `fact_checker.db` and generated PDFs to Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) if you need them to persist.
